<a href="https://colab.research.google.com/github/mariya7040ai/-arabic-english-medqa-uncertainty/blob/dev/mainproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Check the GPU
!nvidia-smi

Sat Oct 17 15:26:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

a Python .gitignore, and a license.

In [2]:
# 2. Reproducibility and logging helpers
import random, datetime
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def log_run(text, path="RUN_LOG.md"):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    with open(path, "a", encoding="utf-8") as f:
        f.write(f"- {ts}: {text}\n")

set_seed(42)
log_run("Day 1 started; seed helper defined (seed 42)")
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())

torch 2.11.0+cu130 | cuda available: True


In [3]:
# 3. Install the FID/KID library
!pip -q install torch-fidelity

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 5.2 MB/s eta 0:00:00


In [4]:
# 4. Sanity check: FID and KID between two real CIFAR-10 splits (values should be small)
# If a flag has changed in your installed version, check the torch-fidelity README.
!fidelity --gpu 0 --fid --kid --input1 cifar10-train --input2 cifar10-val

Creating feature extractor "inception-v3-compat" with features ['2048']
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100% 91.2M/91.2M [00:00<00:00, 99.7MB/s]
Extracting features from input1
100% 170M/170M [45:10<00:00, 62.9kB/s]
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Processing samples:   0% 0/50000 [00:00<?, ?samples/s]/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: User

In [6]:
# 5. Inspect a dataset file. Edit the path and the loader to match the file you downloaded.
import pandas as pd

path = "/content/sample_data/california_housing_test.csv"   # for example a .csv or .json file
df = pd.read_csv(path)               # use pd.read_json(path) for JSON files

print(df.shape)
print(df.columns.tolist())
df.head(5)

(3000, 9)
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-122.05,37.37,27.0,3885.0,661.0,1537.0,606.0,6.6085,344700.0
1,-118.30,34.26,43.0,1510.0,310.0,809.0,277.0,3.5990,176500.0
2,-117.81,33.78,27.0,3589.0,507.0,1484.0,495.0,5.7934,270500.0
3,-118.36,33.82,28.0,67.0,15.0,49.0,11.0,6.1359,330000.0
4,-119.67,36.33,19.0,1241.0,244.0,850.0,237.0,2.9375,81700.0
